### Cài đặt môi trường

In [ ]:
# Cài đặt các thư viện Web API và AI
%pip install fastapi uvicorn python-multipart nest-asyncio transformers torch pillow

!apt-get install -y ssh

### Khởi tạo Server & Model

In [ ]:
%%writefile main.py
from fastapi import FastAPI, UploadFile, File, HTTPException
from transformers import pipeline
from PIL import Image
import io
import torch
import nest_asyncio
import matplotlib.pyplot as plt
from fastapi.staticfiles import StaticFiles
import os

# Tạo thư mục lưu kết quả trên Colab nếu chưa có
os.makedirs("outputs", exist_ok=True)

app = FastAPI()

depth_estimator = pipeline("depth-estimation", model="Intel/dpt-large")

@app.get("/")
def read_root():
    # Giới thiệu hệ thống
    return {
        "project": "2D-to-3D Depth Estimator API",
        "description": "API ước tính độ sâu từ ảnh 2D",
    }

@app.get("/health")
def health_check():
    # Kiểm tra trạng thái
    return {"status": "healthy", "model": "Intel/dpt-large", "device": "cuda" if torch.cuda.is_available() else "cpu"}

@app.post("/predict")
async def predict(file: UploadFile = File(None)):
    # Kiểm tra dữ liệu đầu vào và xử lý lỗi
    if file is None:
        raise HTTPException(status_code=400, detail="Lỗi: Vui lòng tải lên một file ảnh.")

    if not file.content_type.startswith('image/'):
        raise HTTPException(status_code=400, detail="Lỗi: Định dạng file không phải là hình ảnh.")

    try:
        # Đọc ảnh từ request
        contents = await file.read()
        image = Image.open(io.BytesIO(contents)).convert("RGB")

        # Suy luận (Inference)
        result = depth_estimator(image)
        depth_map = result["predicted_depth"]

        # Tạo tên file và lưu heatmap
        pure_filename = os.path.basename(file.filename)
        heatmap_filename = f"heatmap_{pure_filename}.png"
        heatmap_path = os.path.join("outputs", heatmap_filename)
    
        depth_numpy = depth_map.cpu().numpy()
        plt.imsave(heatmap_path, depth_numpy, cmap='magma')

        # Trả kết quả JSON
        return {
            "status": "success",
            "filename": file.filename,
            "heatmap_path": f"/outputs/{heatmap_filename}", # Đường dẫn tải heatmap
            "depth_map_info": {
                "width": depth_map.shape[1],
                "height": depth_map.shape[0],
                "max_depth": float(torch.max(depth_map)),
                "min_depth": float(torch.min(depth_map))
            }
        }
    except Exception as e:
        # Xử lý lỗi trong quá trình suy luận
        return {"status": "error", "message": str(e)}


### Tạo Public URL (Pinggy Tunnel)

In [ ]:
import threading
import uvicorn
import nest_asyncio
import time
import subprocess
import re
import sys

nest_asyncio.apply()

# 1. Giải phóng cổng 8000 
!fuser -k 8000/tcp
!pkill ssh

def run_app():
    uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=False)

# Khởi động server trong thread mới
new_thread = threading.Thread(target=run_app, name="run_app_thread", daemon=True)
new_thread.start()
time.sleep(5)

# 2. Khởi động Tunnel 
proc = subprocess.Popen(
    ['ssh', '-o', 'StrictHostKeyChecking=no', '-p', '443',
     '-R', '80:localhost:8000', 'qr@a.pinggy.io'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1
)

print(" Đang kết nối Pinggy...\n")
pinggy_url = None
output_buffer = ""


start_time = time.time()
while time.time() - start_time < 30: 
    char = proc.stdout.read(1)
    if not char: break
    output_buffer += char

    match = re.search(r'https://[a-z0-9-]+\.(run\.pinggy-free\.link|a\.free\.pinggy\.link)', output_buffer)
    if match:
        pinggy_url = match.group(0)
        print(f"{'='*70}")
        print(f"  LINK TÌM THẤY: {pinggy_url}")
        print(f"  Swagger UI: {pinggy_url}/docs")
        print(f"{'='*70}\n")
        break

if not pinggy_url:
    print("\n  Không tìm thấy link. Log thu được:")
    print(output_buffer)